In [6]:
import torch
from torchvision import transforms
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import login

# 0. Autenticação na Hugging Face (Cole o seu token abaixo)
login(token="")

# 1. Carregando o FairFace via API
print("Carregando o dataset FairFace...")
# Utilizamos a versão pública hospedada na Hugging Face
dataset = load_dataset("HuggingFaceM4/FairFace", "0.25", split='train')

print(f"Total de imagens carregadas: {len(dataset)}")
print(f"Características demográficas disponíveis: {list(dataset.features.keys())}")

# 2. Definindo as transformações da ResNet (Redimensionamento e Normalização)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def preprocess_images(examples):
    """Converte para RGB e aplica as transformações da ResNet."""
    examples['pixel_values'] = [transform(image.convert("RGB")) for image in examples['image']]
    return examples

# 3. Aplicando as transformações sob demanda
dataset.set_transform(preprocess_images)

# 4. Verificação de um exemplo
exemplo = dataset[0]
print("\n--- Verificação do Primeiro Exemplo ---")
print(f"Raça: {exemplo['race']}")
print(f"Gênero: {exemplo['gender']}")
print(f"Idade: {exemplo['age']}")
print(f"Formato do Tensor de Imagem (C, H, W): {exemplo['pixel_values'].shape}")

Carregando o dataset FairFace...
Total de imagens carregadas: 86744
Características demográficas disponíveis: ['image', 'age', 'gender', 'race', 'service_test']

--- Verificação do Primeiro Exemplo ---
Raça: 0
Gênero: 0
Idade: 6
Formato do Tensor de Imagem (C, H, W): torch.Size([3, 224, 224])


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from datasets import load_dataset

# 1. Configuração do Dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo selecionado: {device}")

# 2. Ponte entre o Dataset Hugging Face e o DataLoader PyTorch
class FairFaceDataset(Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset
        # No FairFace, o gênero muitas vezes já vem codificado como int (0: Male, 1: Female)
        # Se vier como string, este dicionário faz a conversão.
        self.gender_to_idx = {'Male': 0, 'Female': 1}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['pixel_values']  # Tensor 3x224x224 (já pré-processado da etapa 1)
        
        # O objetivo do baseline será a classificação de Gênero
        gender = item['gender']
        label = self.gender_to_idx.get(gender, 0) if isinstance(gender, str) else gender
        
        # Mantemos os metadados (race, age) para o cálculo do viés na validação
        race = item['race']
        age = item['age']
        
        return image, torch.tensor(label, dtype=torch.long), race, age

# 3. Inicialização dos DataLoaders
# Carrega os dados de validação usando o subset "0.25" e aplica o preprocessamento da etapa 1
print("Carregando dados de validação...")
val_data = load_dataset("HuggingFaceM4/FairFace", "0.25", split='validation')
val_data.set_transform(preprocess_images) # A função da etapa 1 precisa estar na memória

train_loader = DataLoader(FairFaceDataset(dataset), batch_size=32, shuffle=True)
val_loader = DataLoader(FairFaceDataset(val_data), batch_size=32, shuffle=False)
print("Dataloaders de Treino e Validação configurados.")

# 4. Instanciando o Backbone ResNet-50 (Vanilla)
print("Carregando arquitetura ResNet-50 e pesos pré-treinados no ImageNet...")
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Substituindo a última camada (FC) para classificação binária de Gênero
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model = model.to(device)

# 5. Configuração da Função de Perda (Cross-Entropy clássica) e Otimizador
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# 6. Estrutura do Loop de Treinamento e Validação de Viés
def train_and_evaluate(model, train_loader, val_loader, epochs=1):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        print(f"\nÉpoca {epoch+1}/{epochs} [TREINAMENTO]")
        for inputs, labels, races, ages in tqdm(train_loader, desc="Treinando Batch"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        
        print("Treinamento finalizado. Iniciando Validação Estratificada...")
        model.eval()
        
        # Dicionários para medir a acurácia por subgrupo (Etnia)
        subgroup_correct = {}
        subgroup_total = {}
        
        with torch.no_grad():
            for inputs, labels, races, ages in tqdm(val_loader, desc="Validando Modelo Baseline"):
                inputs, labels = inputs.to(device), labels.to(device)
                
                # Previsão da ResNet
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                
                # Iterando sobre o batch para contar acertos e erros por grupo
                for i in range(len(labels)):
                    race = races[i]
                    # No dataset original, race é int (0-6). Mapeando para legibilidade:
                    race_map = {0: 'White', 1: 'Black', 2: 'Latino_Hispanic', 3: 'East_Asian', 
                                4: 'Southeast_Asian', 5: 'Indian', 6: 'Middle_Eastern'}
                    
                    race_label = race_map.get(race.item() if torch.is_tensor(race) else race, str(race))
                    
                    # Inicializando as chaves caso não existam no dicionário
                    if race_label not in subgroup_total:
                        subgroup_total[race_label] = 0
                        subgroup_correct[race_label] = 0
                    
                    # Contagem
                    subgroup_total[race_label] += 1
                    if preds[i] == labels[i]:
                        subgroup_correct[race_label] += 1
        
        # Imprimindo o Relatório de Viés (Disparidade Demográfica)
        print("\n--- Relatório de Acurácia por Etnia (Baseline ResNet-50) ---")
        for group in subgroup_total.keys():
            acc = subgroup_correct[group] / subgroup_total[group]
            print(f"Etnia: {group:15} | Total: {subgroup_total[group]:4} | Acurácia: {acc*100:.2f}%")
            
    return model

# 7. Disparo da Execução
# (Basta descomentar a linha abaixo para rodar o treinamento e a avaliação real)
model_baseline = train_and_evaluate(model, train_loader, val_loader, epochs=1)

Dispositivo selecionado: cpu
Carregando dados de validação...
Dataloaders de Treino e Validação configurados.
Carregando arquitetura ResNet-50 e pesos pré-treinados no ImageNet...

Época 1/1 [TREINAMENTO]


Treinando Batch:   1%|▌                                                                              | 18/2711 [03:12<7:13:59,  9.67s/it]

In [2]:
#BLOCO - Squeeze-and-Excitation (SE)
import torch
import torch.nn as nn
import time

class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        # Operação de Squeeze: Global Average Pooling (gera um vetor por canal)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        
        # Operação de Excitation: Duas camadas Fully Connected formando o gargalo
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        # Squeeze: (B, C, H, W) -> (B, C, 1, 1) -> (B, C)
        y = self.avg_pool(x).view(b, c)
        # Excitation: Obtém os pesos de recalibração -> (B, C) -> (B, C, 1, 1)
        y = self.fc(y).view(b, c, 1, 1)
        # Escala (multiplica) as features originais pelos pesos
        return x * y.expand_as(x)

In [7]:
#Injeta o Bloco SE na ResNet-50
from torchvision.models.resnet import Bottleneck
import torchvision.models as models
import types

def inject_se_into_resnet(model):
    """
    Varre a ResNet e injeta o SEBlock antes da soma da conexão residual em cada Bottleneck.
    """
    for name, module in model.named_modules():
        if isinstance(module, Bottleneck):
            # O número de canais após as 3 convoluções na ResNet (expansion=4)
            channels = module.conv3.out_channels
            
            # Instancia o bloco SE para a quantidade de canais daquela camada
            se_layer = SEBlock(channel=channels)
            
            original_forward = module.forward
            
            def forward_with_se(x, self_module=module, se_module=se_layer):
                # Guarda a identidade (shortcut)
                identity = self_module.downsample(x) if self_module.downsample is not None else x
                
                # Passa pelas convoluções normais da ResNet
                out = self_module.conv1(x)
                out = self_module.bn1(out)
                out = self_module.relu(out)
                
                out = self_module.conv2(out)
                out = self_module.bn2(out)
                out = self_module.relu(out)
                
                out = self_module.conv3(out)
                out = self_module.bn3(out)
                
                # --- AQUI ACONTECE A MÁGICA DO SENET ---
                # A saída convolucional passa pela recalibração de canais
                out = se_module.to(out.device)(out)
                
                # Soma final com a identidade (shortcut) e ativação
                out += identity
                out = self_module.relu(out)
                
                return out
            
            # Sobrescreve dinamicamente o método da instância
            module.forward = types.MethodType(forward_with_se, module)
            
    return model

# Instanciando a ResNet Híbrida (SE-ResNet)
print("Construindo a arquitetura SE-ResNet-50...")
se_resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
se_resnet = inject_se_into_resnet(se_resnet)

# Adaptando para a classificação binária de gênero
num_ftrs = se_resnet.fc.in_features
se_resnet.fc = nn.Linear(num_ftrs, 2)
se_resnet = se_resnet.to(device) # Usa o device da etapa 1
print("Arquitetura pronta!")

Construindo a arquitetura SE-ResNet-50...


NameError: name 'device' is not defined

In [ ]:
#Treinamento e Registro do Impacto (Tempo vs. Acurácia)
import torch.optim as optim

# Mantemos os hiperparâmetros do baseline
criterion = nn.CrossEntropyLoss()
optimizer_se = optim.Adam(se_resnet.parameters(), lr=0.0001)

print("Iniciando Treinamento do Cenário 2 (SE-ResNet + Cross-Entropy)...")
start_time = time.time()

# Usamos a mesma função de treinamento/validação do Dia 2
se_resnet = train_and_evaluate(se_resnet, train_loader, val_loader, epochs=1)

end_time = time.time()
tempo_total = (end_time - start_time) / 60
print(f"Tempo total de execução do Cenário 2: {tempo_total:.2f} minutos.")